In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.table as table
import pandas as pd

from torch.utils.data import DataLoader
import torch

In [2]:
# Load Preprocessed Data (lemmatization, stopwords, punctuation, etc.)
data = np.genfromtxt('./data/test_data.csv', delimiter='\t', dtype=None, encoding=None, usecols=np.arange(0,2))

reviews, sentiments = zip(*data)

reviews = list(reviews)

data.tolist()[:5]

[('service wa super friendly', 1),
 ('sad little vegetable overcooked', 0),
 ('place wa nice surprise', 1),
 ('golden crispy delicious', 1),
 ('high hope place since burger cooked charcoal grill unfortunately taste fell flat way flat',
  0)]

In [3]:
# Vectorization (the review comments)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1500, min_df=3, max_df=0.6)
vectorized_reviews = vectorizer.fit_transform(reviews).toarray()

In [4]:
vectorized_data = list(zip(vectorized_reviews, sentiments))

In [5]:
from src.Dataset import MyDataset

test_set = MyDataset(vectorized_data, transform=torch.from_numpy)

test_loader = DataLoader(test_set, batch_size=4, shuffle=True, num_workers=1)

In [6]:
from src.Net import Net

net = Net()

PATH = './models/model.pth'
net.load_state_dict(torch.load(PATH))

<All keys matched successfully>

In [8]:
total = 0
correct = 0

all_labels = []
all_outputs = []

# test_loader.__getitem__(0)

with torch.no_grad():
    for data in test_loader:
        reviews, labels = data
        all_labels.append(labels.tolist())
        outputs = net(reviews.float())
        _, predicted = torch.max(outputs.data, 1)
        all_outputs.append(predicted.tolist())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of the network on the {total} test images: {100 * correct / total}")

print(all_labels)
print(all_outputs)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x88 and 368x500)